In [ ]:
# 必要なパッケージの読み込み
library(caret)
library(Metrics)
library(ggplot2)
library(lattice)

# 対象ファイル名一覧
file_names <- c(
  "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
  "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
  "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
  "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)

# データパスの共通部分
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"

# モデルの出力用日付
today <- format(Sys.Date(), "%Y%m%d")

# 評価結果の格納用データフレームの初期化
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_hidden1", "Best_hidden2", "Best_bag", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features")
result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

#--- メインループ ---#
for (file in file_names) {
  filepath <- paste0(base_path, file)
  cat("\n=== Processing:", file, "===\n")
  df_all <- read.csv(filepath, row.names = 1)
  cat("Final dataset size:", dim(df_all)[1], dim(df_all)[2], "\n")

  feature_vars <- setdiff(colnames(df_all), target_vars)

  for (target_var in target_vars) {
    cat("\n---\nTraining model for:", target_var, "on", file, "\n")
    df <- df_all[, c(feature_vars, target_var)]
    df <- df[complete.cases(df), ]

    # モデルチューニング
    tune_grid <- expand.grid(
      hidden1 = c(2, 4, 6),       # 第1層のユニット数
      n.ensemble = c(1, 3, 5)     # アンサンブルの数（デフォルト1）
    )


    ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")

    model <- train(
      formula(paste(target_var, "~ .")),
      data = df,
      method = "monmlp",
      metric = "RMSE",
      trControl = ctrl,
      tuneGrid = tune_grid
    )

    # 交差検証での予測値から該当チューニング条件の行を抽出
    cv_preds <- model$pred
    best_params <- model$bestTune

    for (param in names(best_params)) {
      cv_preds <- cv_preds[cv_preds[[param]] == best_params[[param]], ]
    }

    pred <- cv_preds$pred
    obs <- cv_preds$obs

    R2 <- round(cor(obs, pred)^2, 3)
    RMSE_val <- round(rmse(obs, pred), 3)
    MAE_val <- round(mae(obs, pred), 3)
    RPD_val <- round(sd(obs) / RMSE_val, 3)


    cat("Best params: hidden1 =", model$bestTune$hidden1,
        ", hidden2 =", model$bestTune$hidden2,
        ", bag =", model$bestTune$bag, "\n")
    cat(file, target_var, ": R2 =", R2, ", RMSE =", RMSE_val, ", MAE =", MAE_val, ", RPD =", RPD_val, "\n")

    # 結果保存
    bt <- model$bestTune
    result_matrix[paste0("Best_hidden1_", target_var), file] <- if (!is.null(bt$hidden1)) bt$hidden1 else NA
    result_matrix[paste0("Best_hidden2_", target_var), file] <- if (!is.null(bt$hidden2)) bt$hidden2 else NA
    result_matrix[paste0("Best_bag_", target_var), file]     <- if (!is.null(bt$bag)) bt$bag else NA


    result_matrix[paste0("R2_", target_var), file] <- R2
    result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
    result_matrix[paste0("MAE_", target_var), file] <- MAE_val
    result_matrix[paste0("RPD_", target_var), file] <- RPD_val
    result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
    result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

    # 軸スケール決定関数
    get_axis_limit <- function(values) {
      max_val <- max(values, na.rm = TRUE)
      if (max_val <= 1.5) {
        return(ceiling(max_val / 0.1) * 0.1)
      } else if (max_val <= 5) {
        return(ceiling(max_val / 0.5) * 0.5)
      } else if (max_val <= 30) {
        return(ceiling(max_val / 2) * 2)
      } else {
        return(ceiling(max_val / 5) * 5)
      }
    }

    range_max <- get_axis_limit(c(obs, pred))

    # プロット
    p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
      geom_point(color = "blue", alpha = 0.7) +
      geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
      scale_x_continuous(limits = c(0, range_max)) +
      scale_y_continuous(limits = c(0, range_max)) +
      coord_fixed(ratio = 1) +
      labs(
        title = paste0(target_var, " Prediction (", file, ")"),
        x = "Observed",
        y = "Predicted"
      ) +
      theme_minimal() +
      theme(
        plot.title = element_text(hjust = 0.5, size = 14),
        plot.subtitle = element_text(hjust = 0.5, size = 10)
      ) +
      annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
               label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
      annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
               label = paste0("R² = ", R2))

    # モデル保存
    model_data_bundle <- list(
      model = model,
      training_data = df
    )
    rds_file <- paste0("new20250702_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_monmlp_", today, ".rds")
    saveRDS(model_data_bundle, file = rds_file)

    # PDF出力
    plot_file <- paste0("new20250702_plot_", tools::file_path_sans_ext(file), "_", target_var, "_monmlp_", today, ".pdf")
    pdf(plot_file, width = 6, height = 6)
    print(p)
    dev.off()
  }
}

#--- 結果の保存 ---#
output_file <- paste0("new20250702_monmlp_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\nSummary saved as", output_file, "\n")


Loading required package: ggplot2

Loading required package: lattice


Attaching package: 'Metrics'


The following objects are masked from 'package:caret':

    precision, recall





=== Processing: n_base.csv ===
Final dataset size: 358 11 

---
Training model for: Jsc on n_base.csv 
** Ensemble 1 
0.4701315 
** 0.4701315 

** Ensemble 1 
0.2692108 
** 0.2692108 

** Ensemble 1 
0.2014631 
** 0.2014631 

** Ensemble 1 
0.4126609 
** 0.4126609 

** Ensemble 2 
0.4123389 
** 0.4123389 

** Ensemble 3 
0.4182125 
** 0.4182125 

** Ensemble 1 
0.2751541 
** 0.2751541 

** Ensemble 2 
0.2769219 
** 0.2769219 

** Ensemble 3 
0.2768876 
** 0.2768876 

** Ensemble 1 
0.1982632 
** 0.1982632 

** Ensemble 2 
0.1989311 
** 0.1989311 

** Ensemble 3 
0.2081244 
** 0.2081244 

** Ensemble 1 
0.3958249 
** 0.3958249 

** Ensemble 2 
0.5149168 
** 0.5149168 

** Ensemble 3 
0.4913777 
** 0.4913777 

** Ensemble 4 
0.3774248 
** 0.3774248 

** Ensemble 5 
0.3938038 
** 0.3938038 

** Ensemble 1 
0.2760508 
** 0.2760508 

** Ensemble 2 
0.2915051 
** 0.2915051 

** Ensemble 3 
0.3273034 
** 0.3273034 

** Ensemble 4 
0.2638304 
** 0.2638304 

** Ensemble 5 
0.2922662 
** 0.2922


---
Training model for: Voc on n_base.csv 
** Ensemble 1 
0.2001387 
** 0.2001387 

** Ensemble 1 
0.1603656 
** 0.1603656 

** Ensemble 1 
0.09812216 
** 0.09812216 

** Ensemble 1 
0.2001433 
** 0.2001433 

** Ensemble 2 
0.2274961 
** 0.2274961 

** Ensemble 3 
0.1960271 
** 0.1960271 

** Ensemble 1 
0.1435887 
** 0.1435887 

** Ensemble 2 
0.1411385 
** 0.1411385 

** Ensemble 3 
0.1431577 
** 0.1431577 

** Ensemble 1 
0.103166 
** 0.103166 

** Ensemble 2 
0.1010655 
** 0.1010655 

** Ensemble 3 
0.09812783 
** 0.09812783 

** Ensemble 1 
0.2274948 
** 0.2274948 

** Ensemble 2 
0.2001382 
** 0.2001382 

** Ensemble 3 
0.2001378 
** 0.2001378 

** Ensemble 4 
0.2266943 
** 0.2266943 

** Ensemble 5 
0.2001367 
** 0.2001367 

** Ensemble 1 
0.148889 
** 0.148889 

** Ensemble 2 
0.1450787 
** 0.1450787 

** Ensemble 3 
0.136717 
** 0.136717 

** Ensemble 4 
0.1411493 
** 0.1411493 

** Ensemble 5 
0.1415693 
** 0.1415693 

** Ensemble 1 
0.09192831 
** 0.09192831 

** Ensemble 2


---
Training model for: FF on n_base.csv 
** Ensemble 1 
0.5617158 
** 0.5617158 

** Ensemble 1 
Complex eigenvalues found for method = BFGS 
coefficients for function value 0.5917651  :
 [1]  281.8865377  331.7422980 -198.6058498  357.2977674   96.0988438
 [6]  -49.1299451  456.5602429 -154.1278278 -261.8451067  -43.3040326
[11]  -84.3405431  -89.9069798 -130.4507141   54.2256363 -212.9018473
[16]  -69.4497055 -265.6849625 -222.6692906 -387.8905813 -535.9380521
[21] -240.9300453 -197.9109627  116.8966621 -395.0227868   -8.3229783
[26]   52.8332039  -95.6357143  122.8500986  -98.6200456 -150.1645703
[31]   49.1116612  125.0194952   -0.5234690   -0.2574170   -0.5241502
[36]    0.2429262   -0.4848534
0.5917651 
** 0.5917651 

** Ensemble 1 
0.36778 
** 0.36778 

** Ensemble 1 
0.6295755 
** 0.6295755 

** Ensemble 2 
0.6144424 
** 0.6144424 

** Ensemble 3 
0.6441908 
** 0.6441908 

** Ensemble 1 
0.4371325 
** 0.4371325 

** Ensemble 2 
0.394651 
** 0.394651 

** Ensemble 3 
0.4748962


---
Training model for: PCEmax on n_base.csv 
** Ensemble 1 
0.4365504 
** 0.4365504 

** Ensemble 1 
0.2975659 
** 0.2975659 

** Ensemble 1 
0.2206022 
** 0.2206022 

** Ensemble 1 
0.4235405 
** 0.4235405 

** Ensemble 2 
0.4681035 
** 0.4681035 

** Ensemble 3 
0.4875914 
** 0.4875914 

** Ensemble 1 
0.3101452 
** 0.3101452 

** Ensemble 2 
0.3097046 
** 0.3097046 

** Ensemble 3 
0.3120724 
** 0.3120724 

** Ensemble 1 
0.2714972 
** 0.2714972 

** Ensemble 2 
0.2541048 
** 0.2541048 

** Ensemble 3 
0.218457 
** 0.218457 

** Ensemble 1 
0.4818166 
** 0.4818166 

** Ensemble 2 
0.4789103 
** 0.4789103 

** Ensemble 3 
0.4838617 
** 0.4838617 

** Ensemble 4 
0.4587946 
** 0.4587946 

** Ensemble 5 
0.4764516 
** 0.4764516 

** Ensemble 1 
0.3035454 
** 0.3035454 

** Ensemble 2 
0.2986981 
** 0.2986981 

** Ensemble 3 
0.3111465 
** 0.3111465 

** Ensemble 4 
0.3016798 
** 0.3016798 

** Ensemble 5 
0.3295652 
** 0.3295652 

** Ensemble 1 
0.2800757 
** 0.2800757 

** Ensemble 

...


---
Training model for: FF on n_base_OH.csv 
** Ensemble 1 
Complex eigenvalues found for method = BFGS 
coefficients for function value 0.1728108  :
  [1] -1.3050102007 -0.6190819595 11.7636362084 10.4275926931 -0.7298849275
  [6] -6.7399413662 -1.2159452236 -2.1206482531  0.0980756355  0.4277626805
 [11]  0.2287884721  0.1782447281  0.1131233969 -0.0508004138 -3.1773086300
 [16]  3.8967711041 -0.8067610261  0.1075340545  0.0471290028  0.2932321793
 [21]  0.1131570809  1.3950364345  1.3110389757 -1.1429813802 -2.5144243013
 [26] -0.8788173369 -1.3442535390 -0.7602779545 -0.6960236834 -0.5118331444
 [31]  1.3620938522  0.0531045873 -0.5538915667 -0.6575879814 -0.7351463634
 [36] -0.4647719082 -5.1370679846 -3.9931573260 -5.2604494130 -0.4453926218
 [41] -0.7927552800 -0.2636072292 -0.5250306134 -0.2583396661 -0.8843378824
 [46]  0.0337554285 -0.7003842334 -0.4020758470  0.5993162660  0.1510035586
 [51]  1.6396954768 -0.6995002898 -0.5554619949 -0.6934220138 -0.2217640514
 [56] -0.4497


---
Training model for: PCEmax on n_base_OH.csv 
** Ensemble 1 
Complex eigenvalues found for method = BFGS 
coefficients for function value 0.09365751  :
  [1] -5.845403e-01 -1.781013e+00 -2.319238e+00 -3.562807e-01  4.515197e-01
  [6] -1.446778e+00 -1.824353e+00  7.556942e-02  5.851843e-02 -3.913544e-02
 [11]  3.601956e-02  4.246653e-02  4.567229e-02  2.914338e-02  8.085511e-01
 [16] -8.130429e-01  2.891148e-02 -8.576605e-02  7.328831e-02  2.969715e-03
 [21]  6.389384e-02 -4.474216e-02  3.729351e-01 -1.891102e-01  1.204161e-01
 [26]  1.262453e-01  1.641484e-01  1.144669e-01 -3.565023e-02  8.629525e-02
 [31] -9.974542e-02  3.687339e-02 -4.993334e-02 -7.159530e-02  7.254216e-02
 [36] -6.619608e-01  3.978756e-01  1.482131e-01  4.862353e-01  3.377669e-01
 [41]  8.116029e-02 -1.245206e-01  1.017738e-01  8.321711e-02  1.081672e-01
 [46]  2.171172e-02  1.286830e-01  1.171741e-01  3.659257e-02 -4.035428e-02
 [51] -5.085067e-02  1.367474e-01  1.081550e-01  1.664235e-02  9.284185e-02
 [56] -1

Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_point()`)."



=== Processing: n_base_FP.csv ===
Final dataset size: 358 199 

---
Training model for: Jsc on n_base_FP.csv 
** Ensemble 1 
Complex eigenvalues found for method = BFGS 
coefficients for function value 0.14674  :
  [1]   4.196353523   2.043561669   2.211298284 -12.963642966  -6.536821103
  [6]  12.825609920 -19.511289635  -0.154056569  -0.142785351   0.519987249
 [11]   0.357237488   0.502112106  -0.346110120  -0.386201211  -0.269199708
 [16]   0.277874313  -0.258796006  -0.109906091   0.511259297  -0.273483108
 [21]  -0.277237804   0.325615571   0.467427531  -0.098428867   0.368258423
 [26]   0.356907990  -0.328557444  -0.346570039   0.183697264  -0.357098359
 [31]   0.584082987  -0.121777798  -0.497482770  -0.319788184  -0.306935463
 [36]   0.307083443  -0.258585102  -0.339035489  -0.112644268  -0.522381061
 [41]  -0.266599743  -0.805221495  -0.190274832  -0.073951072  -0.087581437
 [46]  -0.083214503  -0.005673438  -0.186186325  -0.309656139  -0.154233601
 [51]  -0.568678135  -0.25


---
Training model for: Voc on n_base_FP.csv 
